# 04_SES_Feature_Engineering_Final
Production-grade SES Feature Engineering Pipeline

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns",None)

In [2]:
job_skills = pd.read_csv("job_skills.csv")
job_skills_map = pd.read_csv("job_skills_1.csv")
skills_map = pd.read_csv("skills_1.csv")

job_industries = pd.read_csv("job_industries.csv")
industries = pd.read_csv("industries.csv")

linkedin = pd.read_csv("linkedin_job_postings.csv")

postings = pd.read_csv("postings.csv")

salaries = pd.read_csv("salaries.csv")

stack = pd.read_csv(
    "survey_results_public.csv",
    low_memory=False
)

In [3]:
SOFT_SKILLS = [
    "communication",
    "communication skills",
    "teamwork",
    "leadership",
    "customer service",
    "problem solving",
    "problemsolving",
    "collaboration",
    "attention to detail",
    "management",
    "sales",
    "training"
]

In [4]:
all_skills = []

for row in job_skills["job_skills"].dropna():

    skills_list = str(row).split(",")

    all_skills.extend(
        [x.strip().lower() for x in skills_list]
    )

demand_df = (
    pd.Series(all_skills)
    .value_counts()
    .reset_index()
)

demand_df.columns = [
    "skill",
    "linkedin_demand"
]

demand_df = demand_df[
    ~demand_df["skill"].isin(SOFT_SKILLS)
]

demand_df.head(20)

,skill,linkedin_demand
6,time management,142911
9,project management,121563
10,interpersonal skills,100267
11,patient care,99926
13,nursing,88015
16,data analysis,81964
17,microsoft office suite,75531
18,organizational skills,75274
19,inventory management,71911
20,high school diploma,67267


In [5]:
skill_lookup = (
    job_skills_map
    .merge(
        skills_map,
        on="skill_abr",
        how="left"
    )
)

skill_lookup["skill_name"] = (
    skill_lookup["skill_name"]
    .str.lower()
)

skill_lookup.head()

,job_id,skill_abr,skill_name
0,3884428798,MRKT,marketing
1,3884428798,PR,public relations
2,3884428798,WRT,writing/editing
3,3887473071,SALE,sales
4,3887465684,FIN,finance


In [6]:
industry_skill = (
    skill_lookup
    .merge(
        job_industries,
        on="job_id",
        how="left"
    )
)

industry_adoption = (
    industry_skill
    .groupby("skill_name")["industry_id"]
    .nunique()
    .reset_index()
)

industry_adoption.columns = [
    "skill",
    "industry_adoption"
]

industry_adoption.head()

,skill,industry_adoption
0,accounting/auditing,260
1,administrative,263
2,advertising,116
3,analyst,223
4,art/creative,180


In [7]:
country_counts = (
    linkedin["search_country"]
    .value_counts()
)

geo_score = pd.DataFrame({
    "skill": demand_df["skill"],
    "geographic_spread":
    len(country_counts)
})

geo_score.head()

,skill,geographic_spread
6,time management,4
9,project management,4
10,interpersonal skills,4
11,patient care,4
13,nursing,4


In [8]:
salary_jobs = (
    salaries
    .merge(
        job_skills_map,
        on="job_id",
        how="inner"
    )
    .merge(
        skills_map,
        on="skill_abr",
        how="left"
    )
)

salary_jobs["skill_name"] = (
    salary_jobs["skill_name"]
    .str.lower()
)

salary_jobs["salary_score"] = (
    salary_jobs["max_salary"]
    .fillna(0)
    +
    salary_jobs["min_salary"]
    .fillna(0)
) / 2

salary_feature = (
    salary_jobs
    .groupby("skill_name")["salary_score"]
    .mean()
    .reset_index()
)

salary_feature.columns = [
    "skill",
    "salary_premium"
]

salary_feature.head()

,skill,salary_premium
0,accounting/auditing,79443.734424
1,administrative,36685.346739
2,advertising,89163.334432
3,analyst,65816.099677
4,art/creative,65044.877505


In [9]:
from collections import Counter

def extract_counts(series):

    counter = Counter()

    for row in series.dropna():

        for item in str(row).split(";"):

            counter[
                item.strip().lower()
            ] += 1

    return counter

In [10]:
current_usage = Counter()

usage_cols = [

    "LanguageHaveWorkedWith",

    "DatabaseHaveWorkedWith",

    "PlatformHaveWorkedWith",

    "WebframeHaveWorkedWith",

    "ToolsTechHaveWorkedWith",

    "MiscTechHaveWorkedWith"
]

for col in usage_cols:

    current_usage.update(
        extract_counts(stack[col])
    )

current_usage_df = pd.DataFrame(
    current_usage.items(),
    columns=[
        "skill",
        "current_usage"
    ]
)

current_usage_df.head()

,skill,current_usage
0,bash/shell (all shells),20412
1,go,8103
2,html/css,31816
3,java,18239
4,javascript,37492


In [11]:
future_interest = Counter()

future_cols = [

    "LanguageWantToWorkWith",

    "DatabaseWantToWorkWith",

    "PlatformWantToWorkWith",

    "WebframeWantToWorkWith",

    "ToolsTechWantToWorkWith",

    "MiscTechWantToWorkWith"
]

for col in future_cols:

    future_interest.update(
        extract_counts(stack[col])
    )

future_interest_df = pd.DataFrame(
    future_interest.items(),
    columns=[
        "skill",
        "future_interest"
    ]
)

future_interest_df.head()

,skill,future_interest
0,bash/shell (all shells),13744
1,go,13837
2,html/css,20721
3,java,10668
4,javascript,23774


In [18]:
master = demand_df.copy()

master = master.merge(
    industry_adoption,
    on="skill",
    how="left"
)

master = master.merge(
    geo_score,
    on="skill",
    how="left"
)

master = master.merge(
    salary_feature,
    on="skill",
    how="left"
)

master = master.merge(
    current_usage_df,
    on="skill",
    how="left"
)

master = master.merge(
    future_interest_df,
    on="skill",
    how="left"
)

master.fillna(0,inplace=True)

master.head(1000)

,skill,linkedin_demand,industry_adoption,geographic_spread,salary_premium,current_usage,future_interest
0,time management,142911,0.0,4,0.000000,0.0,0.0
1,project management,121563,252.0,4,92119.521639,0.0,0.0
2,interpersonal skills,100267,0.0,4,0.000000,0.0,0.0
3,patient care,99926,0.0,4,0.000000,0.0,0.0
4,nursing,88015,0.0,4,0.000000,0.0,0.0
...,...,...,...,...,...,...,...
995,cash management,2722,0.0,4,0.000000,0.0,0.0
996,policy compliance,2720,0.0,4,0.000000,0.0,0.0
997,guidance,2717,0.0,4,0.000000,0.0,0.0
998,computers,2712,0.0,4,0.000000,0.0,0.0


In [13]:
features = [

    "linkedin_demand",

    "industry_adoption",

    "geographic_spread",

    "salary_premium",

    "current_usage",

    "future_interest"
]

scaler = MinMaxScaler()

master[features] = scaler.fit_transform(
    master[features]
)

In [14]:
master["future_score"] = (

    0.30 * master["future_interest"]

    +

    0.25 * master["linkedin_demand"]

    +

    0.15 * master["current_usage"]

    +

    0.15 * master["industry_adoption"]

    +

    0.10 * master["geographic_spread"]

    +

    0.05 * master["salary_premium"]

)

master.head()

,skill,linkedin_demand,industry_adoption,geographic_spread,salary_premium,current_usage,future_interest,future_score
0,time management,1.000000,0.0,0.0,0.000000,0.0,0.0,0.250000
1,project management,0.850619,0.8,0.0,0.663176,0.0,0.0,0.365814
2,interpersonal skills,0.701602,0.0,0.0,0.000000,0.0,0.0,0.175401
3,patient care,0.699216,0.0,0.0,0.000000,0.0,0.0,0.174804
4,nursing,0.615870,0.0,0.0,0.000000,0.0,0.0,0.153968


In [15]:
master.sort_values(
    "future_score",
    ascending=False
).head(50)

,skill,linkedin_demand,industry_adoption,geographic_spread,salary_premium,current_usage,future_interest,future_score
71,python,0.176104,0.000000,0.0,0.000000,0.819348,0.953735,0.453049
249,javascript,0.066126,0.000000,0.0,0.000000,1.000000,0.905262,0.438110
409,docker,0.043755,0.000000,0.0,0.000000,0.779340,1.000000,0.427840
85,sql,0.154181,0.000000,0.0,0.000000,0.818361,0.852943,0.417182
1693,postgresql,0.011287,0.000000,0.0,0.000000,0.681105,0.914058,0.379205
1,project management,0.850619,0.800000,0.0,0.663176,0.000000,0.000000,0.365814
3138,html/css,0.005745,0.000000,0.0,0.000000,0.848608,0.789011,0.365431
839,typescript,0.022378,0.000000,0.0,0.000000,0.617465,0.770657,0.329411
13081,npm,0.001218,0.000000,0.0,0.000000,0.716580,0.636433,0.298721
6789,amazon web services (aws),0.002533,0.000000,0.0,0.000000,0.591886,0.686924,0.295493


In [16]:
master.to_csv(
    "master_skill_dataset.csv",
    index=False
)

print(
    "Saved : master_skill_dataset.csv"
)

Saved : master_skill_dataset.csv
